# SMITH RIBOMap and STARmap transfer

This notebook reads the real benchmark table used for the RIBOMap transfer analysis and visualizes SMITH against the recorded baselines for each dataset and metric.

[Open the editable source notebook on GitHub](https://github.com/fym0503/SMITH/blob/main/docs/source/tutorials/notebooks/ribomap_section/03_SMITH_RIBOMap_Transfer_source.ipynb)

## Provenance

The code cell verifies the SHA-256 of every bundled result table. These files are copied from completed paper-workspace runs; they are not synthetic replacements for the manuscript outputs.

In [ ]:
from pathlib import Path
import hashlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

def find_repository(start):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "reproducibility").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside a SMITH repository checkout.")

ROOT = find_repository(Path.cwd().resolve())
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 40)

def verify_fixture(relative_path, expected_sha256):
    path = ROOT / "reproducibility" / relative_path
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    assert digest == expected_sha256, f"Checksum mismatch for {path}: {digest}"
    print(f"Verified {relative_path} ({digest[:12]}...)")
    return path


## Analysis

In [ ]:
path = verify_fixture("fixtures/ribomap_benchmark_methods_summary.csv", "ea91769109a2aa473707f1adacc7c2471f80b969daf721cfcc31df2781544e4e")
df = pd.read_csv(path)
accuracy = df[df["metric"].eq("accuracy")].copy()
smith = accuracy[accuracy["method"].eq("SMITH")].sort_values(["dataset", "label"])
display(smith[["dataset", "label", "panel_size", "value_mean", "value_std", "rank"]])
fig, ax = plt.subplots(figsize=(10, 4.5))
for method, group in accuracy.groupby("method"):
    means = group.groupby("dataset")["value_mean"].mean()
    ax.plot(means.index, means.values, marker="o", label=method)
ax.set_ylabel("Mean accuracy"); ax.set_title("RIBOMap/STARmap transfer benchmark")
ax.tick_params(axis="x", rotation=35); ax.legend(frameon=False, ncol=2, fontsize=8)
fig.tight_layout(); plt.show()


## Scope

This is the real completed benchmark summary, not hand-entered tutorial data. Full Figure 4 regeneration additionally requires raw spatial objects and the archived workflows under `reproducibility/workflows/ribomap_transfer/`.